In [4]:
import pandas as pd
import re

df_raw = pd.read_excel(r"C:\Users\husnu\Downloads\клиент_ромашка_остатки.xlsx", header=None)

In [5]:
df = df_raw.iloc[9:17].copy()
df.columns = range(df.shape[1])

print(df)

     0                                  1            2  3    4  5    6  7   \
9   NaN   код продукта - 000000\nПродукт 1  Поставщик 1  1  100  1  100  1   
10  NaN                                NaN        Всего  1  100  1  100  1   
11  NaN   код продукта - 111111\nПродукт 2  Поставщик 2  1  100  1  100  1   
12  NaN                                NaN        Всего  1  100  1  100  1   
13  NaN  код продукта - 1111111\nПродукт 3  Поставщик 2  1  100  1  100  1   
14  NaN                                NaN  Поставщик 3  1  100  1  100  1   
15  NaN                                NaN  Поставщик 4  1  100  1  100  1   
16  NaN                                NaN        Всего  3  300  3  300  3   

     8  9    10  
9   100  3  300  
10  100  3  300  
11  100  3  300  
12  100  3  300  
13  100  3  300  
14  100  3  300  
15  100  3  300  
16  300  9  900  


In [6]:
def extract_product(text):
    if pd.isna(text):
        return pd.Series([None, None])
    
    text = str(text)
    
    code = re.search(r'код продукта\s*-\s*(\d+)', text)
    name = re.search(r'Продукт\s*\d+', text)
    
    return pd.Series([
        code.group(1) if code else None,
        name.group(0) if name else None
    ])

df[['Код продукта', 'Номенклатура']] = df[1].apply(extract_product)

df['Код продукта'] = df['Код продукта'].ffill()
df['Номенклатура'] = df['Номенклатура'].ffill()

print(df[['Код продукта', 'Номенклатура']])

   Код продукта Номенклатура
9        000000    Продукт 1
10       000000    Продукт 1
11       111111    Продукт 2
12       111111    Продукт 2
13      1111111    Продукт 3
14      1111111    Продукт 3
15      1111111    Продукт 3
16      1111111    Продукт 3


In [7]:
df['Поставщик'] = df[2]
df['Поставщик'] = df['Поставщик'].ffill()

print(df[['Поставщик']])

      Поставщик
9   Поставщик 1
10        Всего
11  Поставщик 2
12        Всего
13  Поставщик 2
14  Поставщик 3
15  Поставщик 4
16        Всего


In [8]:
df = df[~df[2].astype(str).str.contains("Всего", na=False)]

print(df)

      0                                  1            2  3    4  5    6  7  \
9   NaN   код продукта - 000000\nПродукт 1  Поставщик 1  1  100  1  100  1   
11  NaN   код продукта - 111111\nПродукт 2  Поставщик 2  1  100  1  100  1   
13  NaN  код продукта - 1111111\nПродукт 3  Поставщик 2  1  100  1  100  1   
14  NaN                                NaN  Поставщик 3  1  100  1  100  1   
15  NaN                                NaN  Поставщик 4  1  100  1  100  1   

      8  9   10 Код продукта Номенклатура    Поставщик  
9   100  3  300       000000    Продукт 1  Поставщик 1  
11  100  3  300       111111    Продукт 2  Поставщик 2  
13  100  3  300      1111111    Продукт 3  Поставщик 2  
14  100  3  300      1111111    Продукт 3  Поставщик 3  
15  100  3  300      1111111    Продукт 3  Поставщик 4  


In [9]:
address_map = {
    "Адрес 1": (3, 4),
    "Адрес 2": (5, 6),
    "Адрес 3": (7, 8),
}

all_parts = []

for addr, (qty_col, sum_col) in address_map.items():
    temp = df[['Код продукта', 'Номенклатура', 'Поставщик']].copy()
    
    temp['Адрес'] = addr
    temp['Кол-во'] = pd.to_numeric(df[qty_col], errors='coerce')
    temp['Сумма'] = pd.to_numeric(df[sum_col], errors='coerce')
    
    all_parts.append(temp)

df_long = pd.concat(all_parts, ignore_index=True)

print(df_long)

   Код продукта Номенклатура    Поставщик    Адрес  Кол-во  Сумма
0        000000    Продукт 1  Поставщик 1  Адрес 1       1    100
1        111111    Продукт 2  Поставщик 2  Адрес 1       1    100
2       1111111    Продукт 3  Поставщик 2  Адрес 1       1    100
3       1111111    Продукт 3  Поставщик 3  Адрес 1       1    100
4       1111111    Продукт 3  Поставщик 4  Адрес 1       1    100
5        000000    Продукт 1  Поставщик 1  Адрес 2       1    100
6        111111    Продукт 2  Поставщик 2  Адрес 2       1    100
7       1111111    Продукт 3  Поставщик 2  Адрес 2       1    100
8       1111111    Продукт 3  Поставщик 3  Адрес 2       1    100
9       1111111    Продукт 3  Поставщик 4  Адрес 2       1    100
10       000000    Продукт 1  Поставщик 1  Адрес 3       1    100
11       111111    Продукт 2  Поставщик 2  Адрес 3       1    100
12      1111111    Продукт 3  Поставщик 2  Адрес 3       1    100
13      1111111    Продукт 3  Поставщик 3  Адрес 3       1    100
14      11

In [10]:
df_long = df_long.dropna(subset=['Кол-во', 'Сумма'])

print(df_long)

   Код продукта Номенклатура    Поставщик    Адрес  Кол-во  Сумма
0        000000    Продукт 1  Поставщик 1  Адрес 1       1    100
1        111111    Продукт 2  Поставщик 2  Адрес 1       1    100
2       1111111    Продукт 3  Поставщик 2  Адрес 1       1    100
3       1111111    Продукт 3  Поставщик 3  Адрес 1       1    100
4       1111111    Продукт 3  Поставщик 4  Адрес 1       1    100
5        000000    Продукт 1  Поставщик 1  Адрес 2       1    100
6        111111    Продукт 2  Поставщик 2  Адрес 2       1    100
7       1111111    Продукт 3  Поставщик 2  Адрес 2       1    100
8       1111111    Продукт 3  Поставщик 3  Адрес 2       1    100
9       1111111    Продукт 3  Поставщик 4  Адрес 2       1    100
10       000000    Продукт 1  Поставщик 1  Адрес 3       1    100
11       111111    Продукт 2  Поставщик 2  Адрес 3       1    100
12      1111111    Продукт 3  Поставщик 2  Адрес 3       1    100
13      1111111    Продукт 3  Поставщик 3  Адрес 3       1    100
14      11

In [11]:
overall = (
    df_long
    .groupby(['Код продукта', 'Номенклатура', 'Поставщик'], as_index=False)
    .agg({
        'Кол-во': 'sum',
        'Сумма': 'sum'
    })
    .rename(columns={
        'Кол-во': 'Общий итог Кол-во',
        'Сумма': 'Общий итог Сумма'
    })
)

print(overall)

  Код продукта Номенклатура    Поставщик  Общий итог Кол-во  Общий итог Сумма
0       000000    Продукт 1  Поставщик 1                  3               300
1       111111    Продукт 2  Поставщик 2                  3               300
2      1111111    Продукт 3  Поставщик 2                  3               300
3      1111111    Продукт 3  Поставщик 3                  3               300
4      1111111    Продукт 3  Поставщик 4                  3               300


In [12]:
df_final = df_long.merge(
    overall,
    on=['Код продукта', 'Номенклатура', 'Поставщик'],
    how='left'
)

print(df_final)

   Код продукта Номенклатура    Поставщик    Адрес  Кол-во  Сумма  \
0        000000    Продукт 1  Поставщик 1  Адрес 1       1    100   
1        111111    Продукт 2  Поставщик 2  Адрес 1       1    100   
2       1111111    Продукт 3  Поставщик 2  Адрес 1       1    100   
3       1111111    Продукт 3  Поставщик 3  Адрес 1       1    100   
4       1111111    Продукт 3  Поставщик 4  Адрес 1       1    100   
5        000000    Продукт 1  Поставщик 1  Адрес 2       1    100   
6        111111    Продукт 2  Поставщик 2  Адрес 2       1    100   
7       1111111    Продукт 3  Поставщик 2  Адрес 2       1    100   
8       1111111    Продукт 3  Поставщик 3  Адрес 2       1    100   
9       1111111    Продукт 3  Поставщик 4  Адрес 2       1    100   
10       000000    Продукт 1  Поставщик 1  Адрес 3       1    100   
11       111111    Продукт 2  Поставщик 2  Адрес 3       1    100   
12      1111111    Продукт 3  Поставщик 2  Адрес 3       1    100   
13      1111111    Продукт 3  Пост

In [13]:
df_final.head(14)

,Код продукта,Номенклатура,Поставщик,Адрес,Кол-во,Сумма,Общий итог Кол-во,Общий итог Сумма
0,000000,Продукт 1,Поставщик 1,Адрес 1,1,100,3,300
1,111111,Продукт 2,Поставщик 2,Адрес 1,1,100,3,300
2,1111111,Продукт 3,Поставщик 2,Адрес 1,1,100,3,300
3,1111111,Продукт 3,Поставщик 3,Адрес 1,1,100,3,300
4,1111111,Продукт 3,Поставщик 4,Адрес 1,1,100,3,300
5,000000,Продукт 1,Поставщик 1,Адрес 2,1,100,3,300
6,111111,Продукт 2,Поставщик 2,Адрес 2,1,100,3,300
7,1111111,Продукт 3,Поставщик 2,Адрес 2,1,100,3,300
8,1111111,Продукт 3,Поставщик 3,Адрес 2,1,100,3,300
9,1111111,Продукт 3,Поставщик 4,Адрес 2,1,100,3,300
